# Week 4 - Baseline Linear Regression

**Core question:** Which non-leaky independent variable bundle best explains `ClosePrice` for Residential SingleFamilyResidence properties using a baseline Linear Regression model?

**Decision supported:** pricing and valuation support for on-market and off-market single-family homes.

**Target:** `ClosePrice`

**Test month:** `2026-05`

**Training window:** `2025-05` through `2026-04`

**Important rule:** Do not use `ListPrice`, `OriginalListPrice`, or any direct price-ratio feature as model inputs.

## 1. Import Packages

Keep the imports small. Week 4 starts with correlation checks and a simple Linear Regression baseline.

In [114]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## 2. Load Week 3 Model-Ready Data

We use the Week 3 train/test files directly. Do not re-split the data in Week 4.

In [115]:
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "outputs" / "week3_preprocessing" / "crmls_sfr_train_X12_2025-05_to_2026-04.csv"
TEST_PATH = PROJECT_ROOT / "outputs" / "week3_preprocessing" / "crmls_sfr_test_2026-05.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (123469, 153)
Test shape: (12012, 153)


In [116]:
target = "ClosePrice"

assert target in train_df.columns
assert target in test_df.columns

print("Target:", target)
print("Train close months:", train_df["close_month"].min(), "to", train_df["close_month"].max())
print("Test close month:", test_df["close_month"].unique())

Target: ClosePrice
Train close months: 2025-05 to 2026-04
Test close month: <ArrowStringArray>
['2026-05']
Length: 1, dtype: str


## 3. Leakage Exclusion Rules

These columns are not allowed as model inputs because they leak the target, identify records, or are unavailable for off-market valuation.

In [117]:
forbidden_features = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "ClosePrice_to_ListPrice_ratio",
    "ListingKey",
    "ListingId",
    "ListingKeyNumeric",
    "CloseDate",
    "close_month",
    "source_month",
    "split",
]

existing_forbidden = [col for col in forbidden_features if col in train_df.columns]
existing_forbidden

['ClosePrice',
 'ListPrice',
 'OriginalListPrice',
 'ClosePrice_to_ListPrice_ratio',
 'ListingKey',
 'ListingId',
 'ListingKeyNumeric',
 'CloseDate',
 'close_month',
 'source_month',
 'split']

## 4. Start With One Key X Variable

First model:

`ClosePrice = intercept + beta_1 * LivingArea_scaled`

Business logic: home size is intrinsic, interpretable, available for on-market and off-market properties, and should have a direct relationship with price.

In [118]:
key_x = "LivingArea_scaled"

assert key_x in train_df.columns

# Compute correlation between LivingArea_scaled and ClosePrice
corr_living_area = train_df[[key_x, target]].corr().loc[key_x, target]
print(f"Correlation between {key_x} and {target}: {corr_living_area:.4f}")

Correlation between LivingArea_scaled and ClosePrice: 0.6192


**Interpretation rules for correlation:**

- `>= 0.70`: strong first X variable
- `0.40 to 0.69`: acceptable but incomplete
- `< 0.40`: weak as a single predictor; location and other features are needed quickly

So, the correlation between LivingArea_scaled and ClosePrice is 0.6192 on the training set. This indicates **a moderate positive relationship**, so LivingArea_scaled is a defensible first independent variable for the baseline model.

**However, the relationship is not strong enough to support valuation decisions by itself.** The model must add other non-leaky features, especially location variables, because California property prices vary substantially by geography.

## 5. Correlation Screening For Candidate X Variables

After the one-variable check, screen other non-leaky numeric features on the training set only. Do not use the test set for feature selection.

In [119]:
candidate_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "BathroomsTotalInteger_scaled",
    "LotSizeSquareFeet_scaled",
    "YearBuilt_scaled",
    "Latitude_scaled",
    "Longitude_scaled",
    "DaysOnMarket_scaled",
    "GarageSpaces_scaled",
    "ParkingTotal_scaled",
    "AssociationFee_scaled",
    "Stories_scaled",
    "City_frequency",
    "PostalCode_frequency",
    "CountyOrParish_frequency",
    "MLSAreaMajor_frequency",
    "HighSchoolDistrict_frequency",
    "ElementarySchoolDistrict_frequency",
    "MiddleOrJuniorSchoolDistrict_frequency",
]

candidate_features = [col for col in candidate_features if col in train_df.columns]

target_corr = (
    train_df[candidate_features + [target]]
    .corr()[target]
    .drop(target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

target_corr

LivingArea_scaled                         0.619250
BathroomsTotalInteger_scaled              0.552276
BedroomsTotal_scaled                      0.321111
AssociationFee_scaled                     0.161273
PostalCode_frequency                     -0.128822
HighSchoolDistrict_frequency              0.119009
Stories_scaled                            0.107042
Longitude_scaled                         -0.086248
GarageSpaces_scaled                       0.055211
City_frequency                            0.048928
CountyOrParish_frequency                  0.039511
MLSAreaMajor_frequency                   -0.033559
DaysOnMarket_scaled                       0.033022
Latitude_scaled                          -0.020068
YearBuilt_scaled                          0.016228
ParkingTotal_scaled                       0.004772
LotSizeSquareFeet_scaled                 -0.001820
ElementarySchoolDistrict_frequency             NaN
MiddleOrJuniorSchoolDistrict_frequency         NaN
Name: ClosePrice, dtype: float6

In [120]:
train_df['ElementarySchoolDistrict'].value_counts()

ElementarySchoolDistrict
Unknown    123469
Name: count, dtype: int64

In [121]:
train_df['MiddleOrJuniorSchoolDistrict'].value_counts()

MiddleOrJuniorSchoolDistrict
Unknown    123469
Name: count, dtype: int64

ElementarySchoolDistrict and MiddleOrJuniorSchoolDistrict were excluded from Week 4 feature candidates because all training rows are coded as Unknown. Their frequency-encoded versions are constant at 1.0, producing undefined correlation with ClosePrice and no predictive value for Linear Regression.

In [122]:
candidate_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "BathroomsTotalInteger_scaled",
    "LotSizeSquareFeet_scaled",
    "YearBuilt_scaled",
    "Latitude_scaled",
    "Longitude_scaled",
    "DaysOnMarket_scaled",
    "GarageSpaces_scaled",
    "ParkingTotal_scaled",
    "AssociationFee_scaled",
    "Stories_scaled",
    "City_frequency",
    "PostalCode_frequency",
    "CountyOrParish_frequency",
    "MLSAreaMajor_frequency",
    "HighSchoolDistrict_frequency"
]


## 6. Multicollinearity Check

Rule for Linear Regression: if two X variables have absolute correlation `>= 0.80`, avoid keeping both unless there is a strong reason.

In [123]:
x_corr = train_df[candidate_features].corr().abs()

high_corr_pairs = []
for i, col_a in enumerate(candidate_features):
    for col_b in candidate_features[i + 1:]:
        corr_value = x_corr.loc[col_a, col_b]
        if corr_value >= 0.80:
            high_corr_pairs.append((col_a, col_b, corr_value))

pd.DataFrame(high_corr_pairs, columns=["feature_a", "feature_b", "abs_corr"]).sort_values(
    "abs_corr", ascending=False
)

,feature_a,feature_b,abs_corr
1,Latitude_scaled,Longitude_scaled,0.925597
0,LivingArea_scaled,BathroomsTotalInteger_scaled,0.839227
2,Latitude_scaled,MLSAreaMajor_frequency,0.828864


The multicollinearity check identified three feature pairs with absolute correlation above 0.80.

1. **Latitude_scaled and Longitude_scaled had very high correlation.**
 Both are retained only in the location bundle because together they represent geographic position. However, linear latitude/longitude is only a rough location proxy and cannot fully capture neighborhood-level price effects.

2. **LivingArea_scaled and BathroomsTotalInteger_scaled were highly correlated.**
LivingArea_scaled is retained as the core size variable because it is the strongest single predictor and has the clearest interpretation. BathroomsTotalInteger_scaled is excluded from the smallest physical-property bundle, but tested later in an expanded structure bundle to determine whether it improves test R².

3. **Latitude_scaled and MLSAreaMajor_frequency were highly correlated.**
 MLSAreaMajor_frequency is excluded from the first location bundle because latitude and longitude provide direct geographic coordinates, while MLSAreaMajor_frequency is only a prevalence encoding. IIt may be tested later in an expanded/full bundle.

 (P.s: MLSAreaMajor_frequency is a frequency-encoded location feature. It represents how common a property's MLS market area is in the training data, not the price level of that area. Because it is highly correlated with Latitude_scaled and only indirectly represents geography, it is excluded from the first location bundle and reserved for expanded feature testing.)


## 7. Baseline Modeling Approach

The baseline models are built one at a time. Each model adds a small group of non-leaky features, then we inspect the training regression output before moving to the next model.

All models use:

`Y = ClosePrice`

Forbidden leakage features:

- `ListPrice`
- `OriginalListPrice`
- `ClosePrice_to_ListPrice_ratio`

The goal is to see whether each added feature group improves explanatory power while staying interpretable.


## 8. Training Regression Setup

Run this setup once. It creates the OLS regression function used by each model below.

Use the narrow statsmodels imports because `import statsmodels.api as sm` fails in this Python 3.13 environment.


In [124]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

train_model_results = []


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    valid = y_true != 0
    return np.mean(np.abs((y_true[valid] - y_pred[valid]) / y_true[valid]))


def mdape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    valid = y_true != 0
    return np.median(np.abs((y_true[valid] - y_pred[valid]) / y_true[valid]))


def run_train_regression(model_name, feature_list):
    X_train = train_df[feature_list]
    X_train = add_constant(X_train)
    y_train = train_df[TARGET]

    model = OLS(y_train, X_train).fit()
    train_pred = model.predict(X_train)

    metrics = {
        "model": model_name,
        "n_features": len(feature_list),
        "train_r2": model.rsquared,
        "train_adj_r2": model.rsquared_adj,
        "train_mape": mape(y_train, train_pred),
        "train_mdape": mdape(y_train, train_pred),
        "features": ", ".join(feature_list),
    }

    train_model_results.append(metrics)

    print(f"Regression result: {model_name}")
    print(f"Train R2: {model.rsquared:.4f}")
    print(f"Adjusted R2: {model.rsquared_adj:.4f}")
    print(f"Train MAPE: {metrics['train_mape']:.4f}")
    print(f"Train MdAPE: {metrics['train_mdape']:.4f}")

    return model.summary()


## 9. Model 1: Size Only

**Question:** How much of `ClosePrice` can be explained by home size alone?

**X variable:**

- `LivingArea_scaled`

This is the minimum baseline. It is simple, interpretable, non-leaky, and available for on-market and off-market homes.

In [125]:
m1_features = [
    "LivingArea_scaled",
]

run_train_regression(
    "M1_size_only",
    m1_features,
)

Regression result: M1_size_only
Train R2: 0.3835
Adjusted R2: 0.3835
Train MAPE: 0.5998
Train MdAPE: 0.4167


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             ClosePrice   R-squared:                       0.383
Model:                            OLS   Adj. R-squared:                  0.383
Method:                 Least Squares   F-statistic:                 7.679e+04
Date:                Mon, 13 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:40:11   Log-Likelihood:            -1.9006e+06
No. Observations:              123469   AIC:                         3.801e+06
Df Residuals:                  123467   BIC:                         3.801e+06
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              1.256e+06   3336.112    376.615      0.000    1.25e+06    1.26e+06
LivingArea_scaled  9.245e+05   3336.112    277.118      0.000    9.18e+05    9.31e+05
==============================================================================
Omnibus:                   246891.237   Durbin-Watson:                   1.935
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       3772441718.758
Skew:                          15.648   Prob(JB):                         0.00
Kurtosis:                     858.752   Cond. No.                         1.00
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Model 1 training result:**

- Training R2: `0.383`

**Interpretation:** Living area alone explains about 38.3% of training-set variation in `ClosePrice`. This is a useful first baseline, but it is not strong enough for valuation decisions because price also depends on structure, location, and market context.

## 10. Model 2: Basic Structure

**Question:** Do basic physical characteristics improve prediction beyond home size?

**X variables:**
`LivingArea_scaled` +  `BedroomsTotal_scaled` + `YearBuilt_scaled` + `LotSizeSquareFeet_scaled`

Bathrooms are excluded here because `BathroomsTotalInteger_scaled` is highly correlated with `LivingArea_scaled`; it is tested separately in Model 3.

In [126]:
m2_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "YearBuilt_scaled",
    "LotSizeSquareFeet_scaled",
]

run_train_regression(
    "M2_basic_structure",
    m2_features,
)

Regression result: M2_basic_structure
Train R2: 0.4403
Adjusted R2: 0.4402
Train MAPE: 0.5403
Train MdAPE: 0.3577


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             ClosePrice   R-squared:                       0.440
Model:                            OLS   Adj. R-squared:                  0.440
Method:                 Least Squares   F-statistic:                 2.428e+04
Date:                Mon, 13 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:40:11   Log-Likelihood:            -1.8946e+06
No. Observations:              123469   AIC:                         3.789e+06
Df Residuals:                  123464   BIC:                         3.789e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                     1.256e+06   3178.773    395.256      0.000    1.25e+06    1.26e+06
LivingArea_scaled         1.172e+06   4335.577    270.307      0.000    1.16e+06    1.18e+06
BedroomsTotal_scaled     -1.962e+05   4224.459    -46.448      0.000   -2.04e+05   -1.88e+05
YearBuilt_scaled         -3.353e+05   3406.277    -98.422      0.000   -3.42e+05   -3.29e+05
LotSizeSquareFeet_scaled -6285.7793   3178.838     -1.977      0.048   -1.25e+04     -55.311
==============================================================================
Omnibus:                   254010.549   Durbin-Watson:                   1.951
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       5172290297.243
Skew:                          16.650   Prob(JB):                         0.00
Kurtosis:                    1005.140   Cond. No.                         2.36
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Model 2 training result:**

- Training R2: `0.440`
- Improvement over Model 1: `+0.057`

**Interpretation:** Bedrooms, year built, and lot size add useful but limited explanatory power beyond living area. Physical structure matters, but structure alone still does not explain enough price variation for a strong valuation model.

## 11. Model 3: Structure + Bathrooms

**Question:** Do bathrooms add useful signal despite multicollinearity with living area?

**X variables:** Model 2 variables plus:

- `BathroomsTotalInteger_scaled`

This model tests whether bathrooms should remain in later bundles.

In [127]:
m3_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "YearBuilt_scaled",
    "LotSizeSquareFeet_scaled",
    "BathroomsTotalInteger_scaled",
]

run_train_regression(
    "M3_structure_with_bathrooms",
    m3_features,
)

Regression result: M3_structure_with_bathrooms
Train R2: 0.4568
Adjusted R2: 0.4568
Train MAPE: 0.5362
Train MdAPE: 0.3653


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             ClosePrice   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.457
Method:                 Least Squares   F-statistic:                 2.077e+04
Date:                Mon, 13 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:40:11   Log-Likelihood:            -1.8928e+06
No. Observations:              123469   AIC:                         3.786e+06
Df Residuals:                  123463   BIC:                         3.786e+06
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                         1.256e+06   3131.398    401.236      0.000    1.25e+06    1.26e+06
LivingArea_scaled             9.162e+05   5967.211    153.545      0.000    9.05e+05    9.28e+05
BedroomsTotal_scaled          -2.62e+05   4297.418    -60.969      0.000    -2.7e+05   -2.54e+05
YearBuilt_scaled             -3.699e+05   3402.645   -108.705      0.000   -3.77e+05   -3.63e+05
LotSizeSquareFeet_scaled     -6098.9103   3131.463     -1.948      0.051   -1.22e+04      38.705
BathroomsTotalInteger_scaled  3.708e+05   6042.442     61.360      0.000    3.59e+05    3.83e+05
==============================================================================
Omnibus:                   253777.791   Durbin-Watson:                   1.952
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       4846869794.021
Skew:                          16.640   Prob(JB):                         0.00
Kurtosis:                     973.067   Cond. No.                         4.07
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

**Model 3 training result:**

- Training R2: `0.457`
- Improvement over Model 2: `+0.017`
- `BathroomsTotalInteger_scaled` p-value: `0.000`

**Interpretation:** Bathrooms add a small but real improvement. The p-value supports an association with `ClosePrice`, but because the training set is very large, the R2 gain is more important than the p-value. The negative bedroom coefficient should not be interpreted causally because bedrooms, bathrooms, and living area are correlated.

## 12. Model 4: Structure + Location

**Question:** Does adding simple location information improve the baseline?

**X variables:** Model 3 variables plus:

`Latitude_scaled` + `Longitude_scaled` + `PostalCode_frequency` + `City_frequency`

Location is essential for California home prices, but these features are still rough proxies. Latitude and longitude are linear approximations, and frequency encodings measure category prevalence, not neighborhood quality.

In [128]:
m4_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "YearBuilt_scaled",
    "LotSizeSquareFeet_scaled",
    "BathroomsTotalInteger_scaled",
    "Latitude_scaled",
    "Longitude_scaled",
    "PostalCode_frequency",
    "City_frequency",
]

run_train_regression(
    "M4_structure_location",
    m4_features,
)

Regression result: M4_structure_location
Train R2: 0.4946
Adjusted R2: 0.4946
Train MAPE: 0.5119
Train MdAPE: 0.3476


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             ClosePrice   R-squared:                       0.495
Model:                            OLS   Adj. R-squared:                  0.495
Method:                 Least Squares   F-statistic:                 1.343e+04
Date:                Mon, 13 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:40:11   Log-Likelihood:            -1.8883e+06
No. Observations:              123469   AIC:                         3.777e+06
Df Residuals:                  123459   BIC:                         3.777e+06
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                         1.381e+06   6220.432    222.090      0.000    1.37e+06    1.39e+06
LivingArea_scaled             9.208e+05   5758.896    159.883      0.000    9.09e+05    9.32e+05
BedroomsTotal_scaled         -2.806e+05   4151.956    -67.590      0.000   -2.89e+05   -2.72e+05
YearBuilt_scaled             -2.315e+05   3665.031    -63.169      0.000   -2.39e+05   -2.24e+05
LotSizeSquareFeet_scaled     -2792.5325   3020.840     -0.924      0.355   -8713.328    3128.263
BathroomsTotalInteger_scaled  3.176e+05   5871.047     54.096      0.000    3.06e+05    3.29e+05
Latitude_scaled              -6.372e+05   8653.467    -73.636      0.000   -6.54e+05    -6.2e+05
Longitude_scaled             -6.932e+05   8976.270    -77.221      0.000   -7.11e+05   -6.76e+05
PostalCode_frequency         -7.964e+07   2.84e+06    -28.076      0.000   -8.52e+07   -7.41e+07
City_frequency                  2.7e+06   3.26e+05      8.287      0.000    2.06e+06    3.34e+06
==============================================================================
Omnibus:                   265671.524   Durbin-Watson:                   1.970
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       6567878884.376
Skew:                          18.529   Prob(JB):                         0.00
Kurtosis:                    1132.290   Cond. No.                     1.54e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.54e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Model 4 training result:**

- Training R2: `0.495`
- Improvement over Model 3: `+0.038`

**Interpretation:** Adding location produces a meaningful improvement, confirming that geography explains price differences beyond physical structure. The coefficients for latitude, longitude, ZIP frequency, and city frequency should not be interpreted causally because they are simplified geographic proxies. The higher condition number suggests multicollinearity among location features, so test-set performance is required before selecting this bundle.

## 13. Model 5: Expanded Non-Leaky Bundle

**Question:** Do additional non-leaky property, market-exposure, and school-context features improve the baseline?

**X variables:** Model 4 variables plus:
 `AssociationFee_scaled` + `Stories_scaled` + `GarageSpaces_scaled` + `DaysOnMarket_scaled` + `HighSchoolDistrict_frequency`

`HighSchoolDistrict_frequency` is included only as a rough frequency-encoded feature. It does not measure school quality.

In [129]:
m5_features = [
    "LivingArea_scaled",
    "BedroomsTotal_scaled",
    "YearBuilt_scaled",
    "LotSizeSquareFeet_scaled",
    "BathroomsTotalInteger_scaled",
    "Latitude_scaled",
    "Longitude_scaled",
    "PostalCode_frequency",
    "City_frequency",
    "AssociationFee_scaled",
    "Stories_scaled",
    "GarageSpaces_scaled",
    "DaysOnMarket_scaled",
    "HighSchoolDistrict_frequency",
]

run_train_regression(
    "M5_expanded_non_leaky",
    m5_features,
)

Regression result: M5_expanded_non_leaky
Train R2: 0.5063
Adjusted R2: 0.5063
Train MAPE: 0.4971
Train MdAPE: 0.3338


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             ClosePrice   R-squared:                       0.506
Model:                            OLS   Adj. R-squared:                  0.506
Method:                 Least Squares   F-statistic:                     9044.
Date:                Mon, 13 Jul 2026   Prob (F-statistic):               0.00
Time:                        23:40:12   Log-Likelihood:            -1.8869e+06
No. Observations:              123469   AIC:                         3.774e+06
Df Residuals:                  123454   BIC:                         3.774e+06
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                         1.376e+06   6365.228    216.177      0.000    1.36e+06    1.39e+06
LivingArea_scaled             9.081e+05   5763.244    157.575      0.000    8.97e+05    9.19e+05
BedroomsTotal_scaled         -2.401e+05   4214.273    -56.966      0.000   -2.48e+05   -2.32e+05
YearBuilt_scaled             -2.026e+05   3708.852    -54.627      0.000    -2.1e+05   -1.95e+05
LotSizeSquareFeet_scaled     -4022.0978   2986.021     -1.347      0.178   -9874.649    1830.453
BathroomsTotalInteger_scaled  3.479e+05   5884.365     59.118      0.000    3.36e+05    3.59e+05
Latitude_scaled              -6.536e+05   8665.416    -75.424      0.000   -6.71e+05   -6.37e+05
Longitude_scaled             -6.975e+05   9024.777    -77.284      0.000   -7.15e+05    -6.8e+05
PostalCode_frequency         -8.476e+07   2.81e+06    -30.181      0.000   -9.03e+07   -7.93e+07
City_frequency                2.525e+06   3.31e+05      7.629      0.000    1.88e+06    3.17e+06
AssociationFee_scaled          6.41e+04   3120.953     20.538      0.000     5.8e+04    7.02e+04
Stories_scaled               -1.609e+05   3368.345    -47.781      0.000   -1.68e+05   -1.54e+05
GarageSpaces_scaled          -1.996e+04   3028.960     -6.588      0.000   -2.59e+04    -1.4e+04
DaysOnMarket_scaled          -3.084e+04   3068.198    -10.053      0.000   -3.69e+04   -2.48e+04
HighSchoolDistrict_frequency  1.909e+05   2.75e+04      6.933      0.000    1.37e+05    2.45e+05
==============================================================================
Omnibus:                   267204.137   Durbin-Watson:                   1.975
Prob(Omnibus):                  0.000   Jarque-Bera (JB):       6894750285.733
Skew:                          18.779   Prob(JB):                         0.00
Kurtosis:                    1160.064   Cond. No.                     1.65e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.65e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Model 5 training result:**

- Training R2: `0.506`
- Improvement over Model 4: `+0.011`

**Interpretation:** The expanded bundle gives only a modest improvement over the location model. `HighSchoolDistrict_frequency` is statistically significant in the training regression, but it is only a frequency encoding and should not be interpreted as school quality. The small R2 gain means final selection must depend on test-set R2, MAPE, and MdAPE, not training R2 alone.

## 14. Training Regression Summary Table

After running Models 1-5, compare the training regression metrics in one table. This table summarizes training fit only; final model selection still requires test-set evaluation.

In [130]:
train_results = pd.DataFrame(train_model_results)
train_results = train_results.sort_values("train_r2", ascending=False).reset_index(drop=True)
train_results

,model,n_features,train_r2,train_adj_r2,train_mape,train_mdape,features
0,M5_expanded_non_leaky,14,0.506316,0.506260,0.497143,0.333760,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
1,M4_structure_location,9,0.494626,0.494589,0.511924,0.347570,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
2,M3_structure_with_bathrooms,5,0.456831,0.456809,0.536245,0.365320,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
3,M2_basic_structure,4,0.440266,0.440248,0.540299,0.357718,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
4,M1_size_only,1,0.383470,0.383465,0.599756,0.416704,LivingArea_scaled


## 15. Training Summary Interpretation

Training R2 increases from Model 1 to Model 5:

- Model 1: `0.383`
- Model 2: `0.440`
- Model 3: `0.457`
- Model 4: `0.495`
- Model 5: `0.506`

The largest conceptual improvement comes from adding location in Model 4. The expanded Model 5 improves training R2 only slightly beyond Model 4, so the extra complexity must be justified by test-set performance.

## 16. Test-Set Evaluation

**Training regression results are useful for understanding coefficients and p-values, but the final Week 4 feature-bundle decision must use the May 2026 test set.**

This function refits each OLS model on the training set, predicts on the test set, and records standardized evaluation metrics.


In [131]:
test_model_results = []


def evaluate_test_set(model_name, feature_list):
    X_train = train_df[feature_list]
    X_train = add_constant(X_train)
    y_train = train_df[TARGET]

    X_test = test_df[feature_list]
    X_test = add_constant(X_test, has_constant="add")
    y_test = test_df[TARGET]

    model = OLS(y_train, X_train).fit()
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    result = {
        "model": model_name,
        "n_features": len(feature_list),
        "train_r2": model.rsquared,
        "test_r2": r2_score(y_test, test_pred),
        "test_mape": mape(y_test, test_pred),
        "test_mdape": mdape(y_test, test_pred),
        "features": ", ".join(feature_list),
    }

    test_model_results.append(result)
    return pd.DataFrame([result])


Run each bundle against the same May 2026 test set. The test set is not used for feature screening or model fitting.


In [132]:
evaluate_test_set("M1_size_only", m1_features)


,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M1_size_only,1,0.38347,0.401926,0.60079,0.417638,LivingArea_scaled


In [133]:
evaluate_test_set("M2_basic_structure", m2_features)


,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M2_basic_structure,4,0.440266,0.4646,0.538086,0.355847,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."


In [134]:
evaluate_test_set("M3_structure_with_bathrooms", m3_features)


,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M3_structure_with_bathrooms,5,0.456831,0.478903,0.533314,0.361794,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."


In [135]:
evaluate_test_set("M4_structure_location", m4_features)


,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M4_structure_location,9,0.494626,0.528094,0.507705,0.338577,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."


In [136]:
evaluate_test_set("M5_expanded_non_leaky", m5_features)


,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M5_expanded_non_leaky,14,0.506316,0.536734,0.496061,0.329826,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."


## 18. Test Evaluation Summary Table

Sort the bundles by test-set `R2`. Lower `MAPE` and `MdAPE` are better.


In [137]:
test_results = pd.DataFrame(test_model_results)
test_results = test_results.sort_values("test_r2", ascending=False).reset_index(drop=True)
test_results

,model,n_features,train_r2,test_r2,test_mape,test_mdape,features
0,M5_expanded_non_leaky,14,0.506316,0.536734,0.496061,0.329826,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
1,M4_structure_location,9,0.494626,0.528094,0.507705,0.338577,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
2,M3_structure_with_bathrooms,5,0.456831,0.478903,0.533314,0.361794,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
3,M2_basic_structure,4,0.440266,0.464600,0.538086,0.355847,"LivingArea_scaled, BedroomsTotal_scaled, YearB..."
4,M1_size_only,1,0.383470,0.401926,0.600790,0.417638,LivingArea_scaled


## 19. Baseline Bundle Decision

Based on the test-set results, **Model 5: Expanded Non-Leaky Bundle** is the best Week 4 Linear Regression baseline.

The Model 5 regression equation is:

The Model 5 regression equation is:

$$
Y =
\beta_0
+ \beta_1 X_1
+ \beta_2 X_2
+ \beta_3 X_3
+ \beta_4 X_4
+ \beta_5 X_5
+ \beta_6 X_6
+ \beta_7 X_7
+ \beta_8 X_8
+ \beta_9 X_9
+ \beta_{10} X_{10}
+ \beta_{11} X_{11}
+ \beta_{12} X_{12}
+ \beta_{13} X_{13}
+ \beta_{14} X_{14}
+ \epsilon
$$

where:

$$
Y = ClosePrice
$$

and:

$$
X_1 = LivingArea\_scaled
$$

$$
X_2 = BedroomsTotal\_scaled
$$

$$
X_3 = YearBuilt\_scaled
$$

$$
X_4 = LotSizeSquareFeet\_scaled
$$

$$
X_5 = BathroomsTotalInteger\_scaled
$$

$$
X_6 = Latitude\_scaled
$$

$$
X_7 = Longitude\_scaled
$$

$$
X_8 = PostalCode\_frequency
$$

$$
X_9 = City\_frequency
$$

$$
X_{10} = AssociationFee\_scaled
$$

$$
X_{11} = Stories\_scaled
$$

$$
X_{12} = GarageSpaces\_scaled
$$

$$
X_{13} = DaysOnMarket\_scaled
$$

$$
X_{14} = HighSchoolDistrict\_frequency
$$

Recovered test-set results:

| Model | Train R2 | Test R2 | Test MAPE | Test MdAPE |
|---|---:|---:|---:|---:|
| M5 expanded non-leaky | 0.506 | 0.537 | 0.496 | 0.330 |
| M4 structure + location | 0.495 | 0.528 | 0.508 | 0.339 |
| M3 structure + bathrooms | 0.457 | 0.479 | 0.533 | 0.362 |
| M2 basic structure | 0.440 | 0.465 | 0.538 | 0.356 |
| M1 size only | 0.383 | 0.402 | 0.601 | 0.418 |

**Decision:** select **Model 5** as the Week 4 baseline feature bundle because it has the **highest test R2** and the **lowest test MAPE and MdAPE**.

**Business implication:** adding location and expanded non-leaky property/market features improves price prediction compared with home size or physical structure alone. However, test MdAPE around 33% means this Linear Regression baseline is not accurate enough for final pricing decisions. It should be treated as a benchmark for later tree-based models, not a production valuation model.

**Limitation:** `HighSchoolDistrict_frequency` is only a frequency encoding and should not be interpreted as school quality. School district enrichment should be revisited in Week 6 using proper geographic joins.
